# 01 Collect

Downloads every input into `data/raw/` and logs it in `data/data_manifest.csv`. All logic lives in `src/reclaim/`; parameters come from `config.yaml`. Every step skips files that already exist.

In [ ]:
from reclaim import config, static, gee
cfg = config.load()

## 1. Habitat maps, land area, boundary

No account needed. Also builds `data/processed/habitat_code_table.csv`, which lists every habitat class with its area in 2010/11 and 2018.

In [ ]:
classes = static.run(cfg)
classes

## 2. Earth Engine: test

One tile of the latest year into `data/raw/_test/`. The first run opens a browser window to sign in. Set `project.ee_project` in `config.yaml` first.

In [ ]:
gee.run(cfg, sources=['s2'], test=True)

In [ ]:
import rasterio, matplotlib.pyplot as plt
from reclaim.config import RAW
with rasterio.open(sorted((RAW / '_test').glob('s2_mndwi_*.tif'))[-1]) as src:
    mndwi, n_obs = src.read(1, masked=True), src.read(2)
fig, ax = plt.subplots(1, 2, figsize=(12, 5))
fig.colorbar(ax[0].imshow(mndwi, cmap='RdBu', vmin=-0.6, vmax=0.6), ax=ax[0]); ax[0].set_title('MNDWI, 80th percentile')
fig.colorbar(ax[1].imshow(n_obs, cmap='viridis'), ax=ax[1]); ax[1].set_title('Cloud-free observations')

## 3. Earth Engine: full download

Sentinel-2 for 2015-2026 and Landsat 8 for 2014-2016, one GeoTIFF per year. Safe to interrupt and rerun.

In [ ]:
gee.run(cfg, sources=['s2', 'landsat'])

## 4. Optional: Sentinel-1 cross-check

In [ ]:
# gee.run(cfg, sources=['s1'])

## 5. What was collected

In [ ]:
import pandas as pd
from reclaim.config import DATA
pd.read_csv(DATA / 'data_manifest.csv')